# ESA-ADB Lite: COPOD

EN: Fast empirical-tail-probability baseline; promising in subset experiments.

中文：快速经验尾概率基线；subset 实验表现有潜力。

EN: This notebook is a thin experiment launcher. The implementation lives in `algorithms.py` and `run_benchmark.py`, so notebook runs and command-line runs stay consistent.

中文：这个 notebook 只是单算法实验入口。算法实现仍在 `algorithms.py` 和 `run_benchmark.py` 中，因此 notebook 与命令行实验保持一致。


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

REPO = Path.cwd()
if not (REPO / "run_benchmark.py").exists():
    REPO = REPO.parent

ALGORITHM = "COPOD"
DATA_PATH = REPO.parent / "data" / "preprocessed"
OUTPUT_DIR = REPO / "results" / "notebooks"

# EN: Good local defaults. For larger runs, change these before running.
# 中文：适合本机的默认配置。大规模实验运行前请修改这些开关。
DATASETS = ["3_months"]
CHANNEL_GROUPS = ["subset"]

# EN: Metric switches.
# 中文：指标开关。
INCLUDE_VUS = False
VUS_MAX_POINTS = 50000
SKIP_OFFICIAL_METRICS = True
SKIP_SLICE_METRICS = False

# EN: Common presets you may copy:
# 中文：可复制使用的常见配置：
# Local quick check / 本机快速验证:
# DATASETS = ["3_months"]; CHANNEL_GROUPS = ["subset"]; INCLUDE_VUS = False
# Cloud subset formal / 云端 subset 正式:
# DATASETS = ["3_months", "10_months", "21_months", "42_months", "84_months"]; CHANNEL_GROUPS = ["subset"]; INCLUDE_VUS = True
# Cloud official classical alignment / 云端官方 classical 对齐:
# DATASETS = ["3_months", "10_months", "21_months", "42_months", "84_months"]; CHANNEL_GROUPS = ["subset"]; INCLUDE_VUS = True
# Cloud target missing algorithms / 云端 target 补跑:
# DATASETS = ["3_months", "10_months", "21_months", "42_months", "84_months"]; CHANNEL_GROUPS = ["target"]; INCLUDE_VUS = True


## Switches / 开关说明

- `DATASETS`: training split list, e.g. `["3_months"]` or all five splits. / 训练集 split 列表。
- `CHANNEL_GROUPS`: `["subset"]`, `["target"]`, or both. / 通道组。
- `INCLUDE_VUS`: add VUS interval metrics. Slower but useful for subsequence anomalies. / 是否计算 VUS 区间指标。
- `SKIP_OFFICIAL_METRICS`: keep `True` unless the official affiliation dependency is available. / 没配好官方依赖时保持 True。
- `SKIP_SLICE_METRICS`: set `False` to produce `slice_metrics.csv`; recommended for research analysis. / 是否跳过异常类型切片指标，研究分析建议 False。

## Run / 运行

Adjust the switches above, then run this cell.


In [ ]:
cmd = [
    sys.executable,
    "run_benchmark.py",
    str(DATA_PATH),
    "--preset",
    "extended",
    "--output-dir",
    str(OUTPUT_DIR),
    "--datasets",
    *DATASETS,
    "--channel-groups",
    *CHANNEL_GROUPS,
    "--algorithms",
    ALGORITHM,
]

if SKIP_OFFICIAL_METRICS:
    cmd.append("--skip-official-metrics")
if SKIP_SLICE_METRICS:
    cmd.append("--skip-slice-metrics")
if INCLUDE_VUS:
    cmd.extend([
        "--include-vus-metrics",
        "--vus-max-points",
        str(VUS_MAX_POINTS),
        "--vus-max-buffer",
        "50",
        "--vus-max-thresholds",
        "30",
    ])

print(" ".join(str(x) for x in cmd))
subprocess.run(cmd, cwd=REPO, check=True)


## Inspect Latest Result / 查看最新结果

This loads the latest notebook run for this algorithm.

下面会读取当前算法最近一次 notebook 输出。


In [ ]:
run_dirs = sorted((OUTPUT_DIR / "extended").glob("*"))
if not run_dirs:
    raise FileNotFoundError("No notebook run found.")

latest = run_dirs[-1]
print(latest)
df = pd.read_csv(latest / "results.csv")
df


In [ ]:
summary_path = latest / "summary_by_algorithm.csv"
if summary_path.exists():
    pd.read_csv(summary_path)


## Slice Metrics / 异常类型切片指标

`slice_metrics.csv` helps answer which algorithm works best for each anomaly type.

`slice_metrics.csv` 用来回答：不同异常类型、长度、局部/全局、多变量/单变量下，哪种算法更好。


In [ ]:
slice_path = latest / "slice_metrics.csv"
if slice_path.exists():
    slice_df = pd.read_csv(slice_path)
    display(slice_df.head(20))
else:
    print("slice_metrics.csv not found. It may have been disabled with SKIP_SLICE_METRICS=True.")


In [ ]:
slice_summary_path = latest / "summary_slice_metrics.csv"
if slice_summary_path.exists():
    summary_slice = pd.read_csv(slice_summary_path)
    display(summary_slice.sort_values(["slice_type", "slice_value", "Point_AUPR"], ascending=[True, True, False]).head(30))
